# Invariance Audit — full run (Colab / any GPU notebook)

Set the 3 variables in the next cell, then **Runtime > Run all**. The notebook does the rest:
gate (500 images) → pilot (200 images + Null D check) → full extraction (2000 images)
→ analysis → site export → zipped downloads.

It **stops itself** with a clear error if the SAE gate rejects or Null D kills the effect.
Needs a GPU runtime (Runtime > Change runtime type > T4).

In [ ]:
REPO_URL = ""  # your git repo URL, OR upload the project folder so audit/ is next to this notebook
HF_TOKEN = ""  # huggingface token with ILSVRC/imagenet-1k access granted
SAVE_TO_DRIVE = True
N_FULL = 2000
N_PILOT = 200
AUTO_CONTINUE = True  # pilot -> full automatically unless a gate fails
DATA = "/content/data/imagenet-val-2k"
OUT = "/content/out"
SITE = "/content/site"

In [ ]:
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    get_ipython().run_line_magic("pip", "-q install open-clip-torch datasets pyarrow")
else:
    get_ipython().run_line_magic("pip", "-q install torch torchvision open-clip-torch datasets huggingface-hub pandas pyarrow pillow tqdm matplotlib scikit-learn")
import torch
print("cuda:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Switch to a GPU runtime first"
if SAVE_TO_DRIVE and IN_COLAB and not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")
assert HF_TOKEN, "Paste your HF_TOKEN in the config cell (needs imagenet-1k access)"
from huggingface_hub import login
login(HF_TOKEN)

In [ ]:
REPO = None
for cand in [".", "..", "/content/repo", "/content/drive/MyDrive/invariance-audit"]:
    if os.path.isdir(os.path.join(cand, "audit")) and os.path.isdir(os.path.join(cand, "interp_core")):
        REPO = os.path.abspath(cand)
        break
if REPO is None:
    assert REPO_URL, "Set REPO_URL or upload the project folder first"
    get_ipython().system(f'git clone "{REPO_URL}" /content/repo')
    REPO = "/content/repo"
sys.path.insert(0, REPO)
print("repo:", REPO)
from audit.extract import set_seed
set_seed(1337)

In [ ]:
if not os.path.isdir(DATA) or len(os.listdir(DATA)) < N_FULL:
    get_ipython().system(f'python "{REPO}/scripts/make_subset.py" --n {N_FULL} --out "{DATA}"')
print(len(os.listdir(DATA)), "images ready")

In [ ]:
from audit.extract import folder_images
from audit.transforms import sweep_grid
sweep_grid(folder_images(DATA, 1)[0][1]).save("/content/sweep.png")
from IPython.display import Image as Show
Show("/content/sweep.png")

**Look at the sweep above.** Rotation must show no black corners, crop must stay centred. If it looks wrong, stop here — no number below is trustworthy.

In [ ]:
from interp_core.loaders import load_model, resolve_hook_name
from interp_core.sae import load_sae, validate_sae
from audit.extract import folder_images, gate_activations
from audit.report import write_gate_card
bundle = load_model("open_clip:ViT-B-32", device="cuda")
sae = load_sae("Prisma-Multimodal/sae-top_k-64-cls_only-layer_9-hook_resid_post", device="cuda")
hook = resolve_hook_name(bundle.spec, sae.hook_layer, sae.hook_component)
print("hook:", hook)
gate = validate_sae(gate_activations(folder_images(DATA, 500), bundle, sae, bundle.preprocess, hook, 500, "auto", "cuda"), sae)
print(gate)
write_gate_card(gate, OUT, hook)
if gate["verdict"] == "reject":
    raise RuntimeError("SAE GATE REJECTED — stopping. Use the attention-head fallback (interp_core.heads), do not train an SAE.")
print("gate passed")

In [ ]:
from audit.extract import extract_to_parquet, load_results
from audit.report import summarize, write_tables
extract_to_parquet(folder_images(DATA, N_PILOT), bundle, sae, bundle.preprocess, OUT + "/pilot", hook_name=hook, pool="auto", device="cuda")
ps = summarize(load_results(OUT + "/pilot"), 32, n_boot=200)
print(write_tables(ps, OUT + "/pilot"))
max_emp = max(max(v["null_d_emptied"]) for v in ps.values())
print("max null-D emptied fraction:", round(max_emp, 3))
if max_emp > 0.5:
    raise RuntimeError("Null D empties >50% of pairs — stopping. Reframe around threshold dependence.")
if not AUTO_CONTINUE:
    raise RuntimeError("Pilot OK. Set AUTO_CONTINUE=True and re-run from here for the full extraction.")
print("pilot passed")

In [ ]:
from audit.report import plot_curves
from audit.sitegen import export_site
out = extract_to_parquet(folder_images(DATA, N_FULL), bundle, sae, bundle.preprocess, OUT, hook_name=hook, pool="auto", device="cuda", gate_n=500)
paths, gacts = out if isinstance(out, tuple) else (out, None)
print("\n".join(paths))
df = load_results(OUT)
summary = summarize(df, 32)
print(write_tables(summary, OUT))
plot_curves(summary, OUT)
export_site(df, dict(folder_images(DATA)), SITE, 32)
for t, v in summary.items():
    print(f"{t}: RII={v['rii']:.3f} [{v['rii_lo']:.3f}, {v['rii_hi']:.3f}]")

In [ ]:
import shutil
shutil.make_archive("/content/invariance-audit-results", "zip", OUT)
shutil.make_archive("/content/invariance-audit-site", "zip", SITE)
if SAVE_TO_DRIVE and os.path.isdir("/content/drive/MyDrive"):
    shutil.copy("/content/invariance-audit-results.zip", "/content/drive/MyDrive/")
    shutil.copy("/content/invariance-audit-site.zip", "/content/drive/MyDrive/")
    print("backed up to Drive")
if IN_COLAB:
    from google.colab import files
    files.download("/content/invariance-audit-results.zip")
    files.download("/content/invariance-audit-site.zip")

## Done — what to do with the downloads

1. Unzip `invariance-audit-results.zip` into local `results/` and re-run `python cli.py analyze --out results/` to confirm identical numbers.
2. Curate `site/data/cases.json`: 3 cases per transform, one hand-written sentence each, then clear the `description_draft` flags.
3. Verify the CLI on a second model (`examples/second_model.md`), deploy `site/` to your domain, publish the write-up.